In [2]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("glass.csv")

if "Id" in df.columns:
    df = df.drop(columns=["Id"])

df["y"] = (df["Type"] == 1).astype(int)
df = df.drop(columns=["Type"])


X = df.drop(columns=["y"]).values
y = df["y"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def predict_proba(X, w, b):
    z = np.dot(X, w) + b
    return sigmoid(z)

def loss(y, p):
    epsilon = 1e-8  # to avoid log(0)
    return -np.mean(y*np.log(p + epsilon) + (1-y)*np.log(1-p + epsilon))

def update_weights(X, y, w, b, lr):
    p = predict_proba(X, w, b)
    error = p - y
    w = w - lr * np.dot(X.T, error) / len(y)
    b = b - lr * np.mean(error)
    return w, b

w = np.zeros(X_train.shape[1])
b = 0.0
lr = 0.1
epochs = 100

for _ in range(epochs):
    w, b = update_weights(X_train, y_train, w, b, lr)


train_probs = predict_proba(X_train, w, b)
test_probs = predict_proba(X_test, w, b)

train_loss = loss(y_train, train_probs)
test_loss = loss(y_test, test_probs)


def predict_label(p, threshold=0.5):
    return (p >= threshold).astype(int)

test_preds_05 = predict_label(test_probs, 0.5)
test_preds_07 = predict_label(test_probs, 0.7)

accuracy_05 = np.mean(test_preds_05 == y_test)
accuracy_07 = np.mean(test_preds_07 == y_test)

print("Final Weights:\n", w)
print("Final Bias:\n", b)
print("\nTraining Loss:", train_loss)
print("Test Loss:", test_loss)
print("\nTest Accuracy (threshold=0.5):", accuracy_05)
print("Test Accuracy (threshold=0.7):", accuracy_07)

Final Weights:
 [ 0.10796659 -0.15606355  0.63480916 -0.67204019  0.07051998 -0.08936205
 -0.19026934 -0.16364966 -0.1099809 ]
Final Bias:
 -0.7346250485229616

Training Loss: 0.4977756010941282
Test Loss: 0.41082663160955274

Test Accuracy (threshold=0.5): 0.8604651162790697
Test Accuracy (threshold=0.7): 0.7209302325581395



UCS761 – Deep Learning Lab 3
Logistic Regression as a Soft Decision Model
1. Loading the Dataset

In this step, we load the glass dataset into a DataFrame using pandas. Each row represents one glass sample and each column represents a feature or the class label.

This step is necessary because the model can only work with numerical data arranged in a structured format.

2. Inspecting the Data

After loading the dataset, we check the shape, column names, and first few rows.

The output column is “Type”.
All columns are numeric.
There is an ID column which should not be used for training because it does not carry meaningful information for prediction.

This step is important because models do not understand the meaning of data — they only see numbers. If we include unnecessary columns, the model may learn incorrect patterns.

3. Converting to Binary Classification

The dataset originally contains multiple glass types. For this lab, we convert it into a binary problem:

Type 1 → 1

All other types → 0

We create a new column y for this and remove the original Type column.

This is done because our implementation of logistic regression handles binary classification.

4. Separating Inputs and Outputs

We separate:

X → feature values

y → target labels

This is required because the model learns the relationship between input features and output labels.

5. Train-Test Split

We split the data into training (80%) and testing (20%).

The model learns using the training set and is evaluated using the test set.

This is important because testing on training data would give an unrealistic measure of performance.

6. Feature Scaling

We scale the features using StandardScaler.

Glass features have different numerical ranges. If we do not scale them, some features may dominate others, and the sigmoid function may saturate.

Scaling ensures stable learning.

7. Sigmoid Function

The sigmoid function converts the linear score into a value between 0 and 1.

This represents the probability that a sample belongs to the positive class.

Unlike the perceptron’s step function, sigmoid gives confidence information.

8. Forward Computation

We compute:

z = Xw + b
p = sigmoid(z)

Here:

z is the score (evidence)

p is the probability

This gives us how confident the model is about each prediction.

9. Loss Function

We use binary cross-entropy loss.

This measures how wrong the predicted probabilities are.

If the model is confidently wrong, the penalty is large.
If it is uncertain, the penalty is smaller.

10. Updating Weights

We compute the error (p − y) and update the weights and bias using gradient descent.

The learning rate controls how large each update step is.

This gradually reduces the loss over time.

11. Decision Threshold

We convert probabilities into final labels using a threshold.

At 0.5 → normal decision boundary.
At 0.7 → stricter decision.

In glass quality control, using 0.7 is safer because we only classify as positive when the model is very confident. This reduces false positives.

Final Paragraph

Logistic regression differs from the perceptron because it outputs probabilities instead of hard 0 or 1 values. The perceptron uses a step function, while logistic regression uses the sigmoid function, which provides confidence levels. Logistic regression also uses cross-entropy loss, making learning smoother and more stable.

However, logistic regression is still a linear model. It cannot solve non-linearly separable problems like XOR. For more complex patterns, deeper neural networks are required.